### Import Library & dataset ###

In [1]:
import numpy as np
import pandas as pd
df = pd.read_csv("dimGame_raw.csv", sep=',', encoding='utf-8')
df.head(10)

,game_id,game_name,genre,publisher,platform
0,NaN,Grand Theft Auto IV,Action / Shooter / Racing,Rockstar,PS3
1,NaN,Red Dead Redemption 2,Action / Adventure / Unique,Rockstar Games,PS4
2,NaN,Red Dead Online,Action / Adventure,Rockstar Games,PS4
3,NaN,Grand Theft Auto 3,Shooter / Racing / Simulator / Adventure,Rockstar Games,PS3
4,NaN,Grand Theft Auto V,Action / Adventure,Rockstar Games,PS3
5,NaN,Grand Theft Auto V,Action / Adventure,Rockstar Games,PS4
6,NaN,Baldur's Gate 3,Role playing games,Larian Studios Games Ltd,PS5 / PS4
7,NaN,ELDEN RING PS4 & PS5,Role playing games,BANDAI NAMCO ENTERTAINMENT EUROPE,PS5 / PS4
8,NaN,Uncharted 2: Among Thieves™,Shooter / Platform / Adventure,Sony Interactive Entertainment Europe,PS3
9,NaN,Mass Effect™ 2,Role playing games,EA Swiss Sarl,PS3


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12068 entries, 0 to 12067
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   game_id    0 non-null      float64
 1   game_name  12068 non-null  str    
 2   genre      12068 non-null  str    
 3   publisher  12068 non-null  str    
 4   platform   12068 non-null  str    
dtypes: float64(1), str(4)
memory usage: 1.2 MB


### Cek Null dan Persentase ###

In [3]:
print(f"Jumlah data asli: {len(df)}\n")
print("\nJumlah baris duplikat:", df.duplicated().sum())
pd.DataFrame({
    'missing_value' : df.isnull().sum(),
    'missing_value_percentage' : df.isnull().sum() / len(df) * 100
})

Jumlah data asli: 12068


Jumlah baris duplikat: 42


,missing_value,missing_value_percentage
game_id,12068,100.0
game_name,0,0.0
genre,0,0.0
publisher,0,0.0
platform,0,0.0


### Data Wrangling ###

In [4]:
# hapus spasi dan '/' lalu buat atomik
df['platform'] = df['platform'].astype(str).str.split(r'\s*/\s*')
df_exploded = df.explode('platform')
df_exploded['platform'] = df_exploded['platform'].str.strip()
df_exploded.head(10)

,game_id,game_name,genre,publisher,platform
0,NaN,Grand Theft Auto IV,Action / Shooter / Racing,Rockstar,PS3
1,NaN,Red Dead Redemption 2,Action / Adventure / Unique,Rockstar Games,PS4
2,NaN,Red Dead Online,Action / Adventure,Rockstar Games,PS4
3,NaN,Grand Theft Auto 3,Shooter / Racing / Simulator / Adventure,Rockstar Games,PS3
4,NaN,Grand Theft Auto V,Action / Adventure,Rockstar Games,PS3
5,NaN,Grand Theft Auto V,Action / Adventure,Rockstar Games,PS4
6,NaN,Baldur's Gate 3,Role playing games,Larian Studios Games Ltd,PS5
6,NaN,Baldur's Gate 3,Role playing games,Larian Studios Games Ltd,PS4
7,NaN,ELDEN RING PS4 & PS5,Role playing games,BANDAI NAMCO ENTERTAINMENT EUROPE,PS5
7,NaN,ELDEN RING PS4 & PS5,Role playing games,BANDAI NAMCO ENTERTAINMENT EUROPE,PS4


In [5]:
#normalisasi penamaan
df_exploded['game_name'] = df_exploded['game_name'].astype(str).str.title()
df_exploded['genre'] = df_exploded['genre'].astype(str).str.title()
df_exploded['publisher'] = df_exploded['publisher'].astype(str).str.title()
#hapus simbol di nama game
df_exploded['game_name'] = df_exploded['game_name'].str.replace(r'[®™©]', '', regex=True)
df_exploded['game_name'] = df_exploded['game_name'].str.replace(r'\s+', ' ', regex=True).str.strip()

df_exploded.iloc[59:70]


,game_id,game_name,genre,publisher,platform
48,NaN,Tom Clancy'S Splinter Cell Hd,Action / Adventure,Ubisoft Entertainment Sa,PS3
49,NaN,Persona 4 Golden,Role Playing Games,Nis America,PS Vita
50,NaN,Metal Gear Solid V: The Phantom Pain,Action,Konami,PS3
51,NaN,Metal Gear Solid V: The Phantom Pain,Action,Konami,PS4
52,NaN,Blue Prince,Puzzle / Strategy / Adventure / Indie,Raw Fury,PS5
53,NaN,Forza Horizon 5 Standard Edition,Unknown,Microsoft Corporation,PS5
54,NaN,Final Fantasy Vi,Role Playing Games,Square Enix Ltd,PS4
55,NaN,The Witcher 3: Wild Hunt,Role Playing Games,Cd Projekt,PS5
55,NaN,The Witcher 3: Wild Hunt,Role Playing Games,Cd Projekt,PS4
56,NaN,Rimworld Console Edition,Simulation,Double Eleven Limited,PS4


In [6]:
#normalisasi kolom platform
platform_unik = df_exploded['platform'].unique()
print("Daftar Platform:")
print(platform_unik)

Daftar Platform:
<ArrowStringArray>
[                           'PS3',                            'PS4',
                            'PS5',                        'PS Vita',
                            'PSP',                   'THEKLA INC.,',
                  'PlayStation 4',                  'PlayStation 3',
 'Sony Interactive Entertainment',                       '3909 LLC',
       'WARNER BROS. INTERACTIVE',        'Klei Entertainment Inc.',
      'FRONTIER DEVELOPMENTS PLC',             'OUTRIGHT GAMES LLC',
              'Owlcat Games Ltd.',               'Kasedo Games Ltd',
         'DAEDALIC ENTERTAINMENT',                         'Action',
            'Focus Entertainment',                  'PlayStation 5',
                      'ROGUESIDE',            'Toplitz Productions',
                    'NIS America',                       'NACON SA']
Length: 24, dtype: str


In [7]:
#koreksi untuk platform yang beda ejaan
corec_platform = {
    'PlayStation 3': 'PS3',
    'PlayStation 4': 'PS4',
    'PlayStation 5': 'PS5',
}
df_exploded['platform'] = df_exploded['platform'].replace(corec_platform)
#hapus duplicate speisifik
platform_valid = ['PS3','PS5','PS4','PSP','PS Vita']
platform_anomaly = df_exploded.loc[~df_exploded['platform'].isin(platform_valid)]
print("Platform Anomali:")
print(platform_anomaly[['game_name', 'platform']])
#drop yang 'warhammer 40' soalnya isi nya dupe semua
df_exploded = df_exploded.drop(index = [828,1320,1368,1499,1864,3264])
index = [293,294,367,427,489,490,771,1154,1406,2203,2463,3010,]
value = ['PS4','PS5','PSP','PS Vita','PS4','PS5','PS4','PS4','PS4','PSP','PS4','PSP']
df_exploded.loc[index, 'platform'] = value

Platform Anomali:
                                     game_name                        platform
293                                      Braid                    THEKLA INC.,
294                                      Braid                    THEKLA INC.,
367                 Ratchet & Clank: La Taille  Sony Interactive Entertainment
427                                     Papers                        3909 LLC
489   Hogwarts Legacy : L'Héritage De Poudlard        WARNER BROS. INTERACTIVE
490   Hogwarts Legacy : L'Héritage De Poudlard        WARNER BROS. INTERACTIVE
771                                  Invisible         Klei Entertainment Inc.
828                               Warhammer 40       FRONTIER DEVELOPMENTS PLC
1154                                Paw Patrol              OUTRIGHT GAMES LLC
1320                              Warhammer 40               Owlcat Games Ltd.
1368                              Warhammer 40                Kasedo Games Ltd
1406                   Les Piliers

In [8]:
platform_unik = df_exploded['platform'].unique()
print("Daftar Platform:")
print(platform_unik)

Daftar Platform:
<ArrowStringArray>
['PS3', 'PS4', 'PS5', 'PS Vita', 'PSP']
Length: 5, dtype: str


In [9]:
# Isi game_id 
df_exploded['game_id'] = pd.to_numeric(df_exploded['game_id'], errors='coerce')
df_exploded['game_id'] = range(1, len(df_exploded) + 1)
df_exploded.iloc[59:70]

,game_id,game_name,genre,publisher,platform
48,60,Tom Clancy'S Splinter Cell Hd,Action / Adventure,Ubisoft Entertainment Sa,PS3
49,61,Persona 4 Golden,Role Playing Games,Nis America,PS Vita
50,62,Metal Gear Solid V: The Phantom Pain,Action,Konami,PS3
51,63,Metal Gear Solid V: The Phantom Pain,Action,Konami,PS4
52,64,Blue Prince,Puzzle / Strategy / Adventure / Indie,Raw Fury,PS5
53,65,Forza Horizon 5 Standard Edition,Unknown,Microsoft Corporation,PS5
54,66,Final Fantasy Vi,Role Playing Games,Square Enix Ltd,PS4
55,67,The Witcher 3: Wild Hunt,Role Playing Games,Cd Projekt,PS5
55,68,The Witcher 3: Wild Hunt,Role Playing Games,Cd Projekt,PS4
56,69,Rimworld Console Edition,Simulation,Double Eleven Limited,PS4


In [10]:
#one-hot encoding untuk genre dan masalah tanggal
df_exploded['genre'] = df_exploded['genre'].str.replace(r'[a-zA-Z]{3} \d{1,2}, \d{4}', '', regex=True)
df_exploded['genre'] = df_exploded['genre'].str.replace(' / ', '/')
df_exploded['genre'] = df_exploded['genre'].str.replace(r'//+', '/', regex=True).str.strip('/')
#One-Hot Encoding
df_genre_ohe = df_exploded['genre'].str.get_dummies(sep='/')
df_exploded = pd.concat([df_exploded, df_genre_ohe], axis=1)

#menggabungkan kolom Role Playing Games dan Role-Playing (Rpg) menjadi satu kolom RPG
df_exploded['RPG'] = df_exploded[['Role Playing Games', 'Role-Playing (Rpg)']].max(axis=1)
df_exploded = df_exploded.drop(columns=['Role Playing Games', 'Role-Playing (Rpg)'])

#kolom kosong & tidak guna
df_exploded = df_exploded.drop(columns=['', ' '], errors='ignore')
df_exploded = df_exploded.drop(columns=['genre'], errors='ignore')
df_exploded.head()

,game_id,game_name,publisher,platform,Action,Adult,Adventure,Arcade,Beat 'Em Up,Brain Training,...,Sport,Sports,Strategy,Tactical,Trivia,Turn-Based Strategy (Tbs),Unique,Unknown,Visual Novel,RPG
0,1,Grand Theft Auto Iv,Rockstar,PS3,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Red Dead Redemption 2,Rockstar Games,PS4,1,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2,3,Red Dead Online,Rockstar Games,PS4,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,Grand Theft Auto 3,Rockstar Games,PS3,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Grand Theft Auto V,Rockstar Games,PS3,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
df_exploded.to_csv("dimGame_cleaned.csv", index=False, encoding='utf-8')